# Post-Hoc Analysis Pipeline
### Katherine (ZhaoYu) Tu — Master Thesis
**University of Amsterdam, 2026**

This notebook contains all post-hoc analyses for SRQ1 and SRQ3.
Run cells in order. All outputs are saved to the `results/` folder.

---
**SRQ1 analyses:**
- Accuracy summary table
- Confusion matrices (per model, per task), deeper analysis check `post_hoc_deep_analysis.ipynb`
    1. Relative major/minor confusion analysis
    2. Enharmonic equivalence errors
    3. Accidental direction errors (sharp vs flat confusion)
    4. Error distance in the circle of fifths
    5. Per-model error overlap
- Per-class accuracy (which keys/time sigs are hardest)
- Cross-model agreement (Definition A)

**SRQ3 analyses:**
- Accuracy summary table
- McNemar's test (PDF vs PNG, per model per task)
- Format effect direction (when they disagreed, which format was right?)
- Confusion matrices (per model, per task, per format)
- Per-class accuracy delta (PDF vs PNG)
- Cross-format consistency (Definition B)

---
## Setup

This notebook expects dependencies from `requirements.txt` to be installed, and the result JSONLs to exist in `../results/`. See the project README for full setup.

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
from pathlib import Path

# Resolve paths relative to the repo root (which is one level up from notebooks/)
REPO_ROOT   = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"

SRQ1_FILE = RESULTS_DIR / "srq1_results_fixed.jsonl"
SRQ2_FILE = RESULTS_DIR / "srq2_cot_results.jsonl"
SRQ3_FILE = RESULTS_DIR / "srq3_results.jsonl"

# Sanity check
assert RESULTS_DIR.exists(), f"Cannot find {RESULTS_DIR}. Run from the notebooks/ folder or the repo root."

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
MODELS = ["claude-sonnet-4-6", "gemini-2.5-flash", "gpt-5.4"]
TASKS  = ["key_signature", "time_signature"]

# ── SHARED KEY NORMALISATION HELPERS ─────────────────────────────────────────
# Copied from pipeline notebook so this notebook is self-contained.

def dcml_key_to_standard(dcml_key):
    """'F' → ('f','major'),  'f' → ('f','minor'),  'Bb' → ('bb','major')"""
    if not dcml_key:
        return None, None
    quality = "major" if dcml_key[0].isupper() else "minor"
    return dcml_key.lower(), quality

def normalise_key_for_comparison(raw_key):
    """Normalise a model response string to (note, quality) tuple."""
    if not raw_key:
        return None, None
    raw = raw_key.strip()
    if len(raw) <= 3 and " " not in raw:
        return dcml_key_to_standard(raw)
    raw_lower = raw.lower()
    if "major" in raw_lower or "maj" in raw_lower:
        quality = "major"
    elif "minor" in raw_lower or "min" in raw_lower:
        quality = "minor"
    else:
        quality = "major"
    note = raw_lower
    for word in ["major", "minor", "maj", "min"]:
        note = note.replace(word, "")
    note = note.strip().rstrip("-").strip()
    note = note.replace("flat", "b").replace("sharp", "#")
    note = note.replace("\u266d", "b").replace("\u266f", "#")
    note = note.replace("-", "").replace(" ", "")
    return note, quality

def keys_match(ground_truth_val, parsed_answer):
    if not ground_truth_val or not parsed_answer:
        return False
    gt_note, gt_quality     = dcml_key_to_standard(str(ground_truth_val))
    pred_note, pred_quality = normalise_key_for_comparison(parsed_answer)
    return gt_note == pred_note and gt_quality == pred_quality

def normalise_for_sklearn(val, question_type):
    """Normalise to a single comparable string for sklearn."""
    if not val or str(val).strip() == "":
        return "unknown"
    val = str(val).strip()
    if question_type == "key_signature":
        if len(val) <= 3 and " " not in val:
            note, quality = dcml_key_to_standard(val)
            if note and quality:
                return f"{note} {quality}"
        note, quality = normalise_key_for_comparison(val)
        if note and quality:
            return f"{note} {quality}"
        return val.lower()
    return val.lower().strip()

# ── LOADER ────────────────────────────────────────────────────────────────────

def load_jsonl(filepath):
    """Load a JSONL file into a DataFrame."""
    records = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} entries from {Path(filepath).name}")
    print(f"Error rate: {(df['model_response'] == 'ERROR').mean():.1%}")
    return df

# ── METRIC CALCULATOR ─────────────────────────────────────────────────────────

def compute_metrics(df_subset, question_type):
    """Compute accuracy + macro P/R/F1 for a results subset."""
    df_clean = df_subset[df_subset["model_response"] != "ERROR"].copy()
    if df_clean.empty:
        return {"n": 0, "n_errors": len(df_subset),
                "accuracy": None, "precision": None,
                "recall": None, "f1": None}
    y_true = df_clean["ground_truth"].apply(
        lambda x: normalise_for_sklearn(x, question_type))
    y_pred = df_clean["parsed_answer"].fillna("").apply(
        lambda x: normalise_for_sklearn(x, question_type))
    accuracy = df_clean["correct"].mean()
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    return {"n": len(df_clean),
            "n_errors": len(df_subset) - len(df_clean),
            "accuracy": round(accuracy, 4),
            "precision": round(p, 4),
            "recall": round(r, 4),
            "f1": round(f, 4)}

print("Cell 1 done — imports, config, helpers loaded.")

---
## Cell 2 — Load results

In [ ]:
df1 = load_jsonl(SRQ1_FILE)
df2 = load_jsonl(SRQ2_FILE)
df3 = load_jsonl(SRQ3_FILE)

# Quick sanity checks
print(f"\nSRQ1 — models:  {df1['model'].unique().tolist()}")
print(f"SRQ1 — tasks:   {df1['question_type'].unique().tolist()}")
print(f"SRQ1 — formats: {df1['input_type'].unique().tolist()}")

print(f"\nSRQ3 — models:  {df3['model'].unique().tolist()}")
print(f"SRQ3 — tasks:   {df3['question_type'].unique().tolist()}")
print(f"SRQ3 — formats: {df3['input_type'].unique().tolist()}")
print(f"SRQ3 — variants:{df3['prompt_variant'].unique().tolist()}")

---
# SRQ1 Analysis
## Cell 3 — SRQ1: Accuracy summary table

In [ ]:
print("=" * 65)
print("SRQ1 — Accuracy Summary (Zero-shot, PDF)")
print("=" * 65)
print(f"{'model':<25} {'task':<20} {'n':>4} {'errors':>6} "
      f"{'accuracy':>9} {'precision':>10} {'recall':>7} {'f1':>6}")
print("-" * 90)

srq1_summary = []

for model in MODELS:
    for task in TASKS:
        sub = df1[(df1["model"] == model) & (df1["question_type"] == task)]
        m   = compute_metrics(sub, task)
        print(f"{model:<25} {task:<20} {m['n']:>4} {m['n_errors']:>6} "
              f"{m['accuracy']:>9.3f} {m['precision']:>10.3f} "
              f"{m['recall']:>7.3f} {m['f1']:>6.3f}")
        srq1_summary.append({"model": model, "task": task, **m})

print("\nAll metrics use macro averaging.")

srq1_summary_df = pd.DataFrame(srq1_summary)
srq1_summary_df.to_csv(RESULTS_DIR / "srq1_summary.csv", index=False)
print(f"Saved to srq1_summary.csv")

---
## Cell 4 — SRQ1: Confusion matrices

In [ ]:
print("=" * 65)
print("SRQ1 — Confusion Matrices")
print("=" * 65)

fig, axes = plt.subplots(
    len(MODELS), len(TASKS),
    figsize=(20, 8 * len(MODELS))
)

fig.suptitle(
    "SRQ1 — Confusion Matrices (Zero-shot, PDF)",
    fontsize=14, fontweight="bold", y=1.01
)

for i, model in enumerate(MODELS):
    for j, task in enumerate(TASKS):
        df_sub = df1[
            (df1["model"] == model) &
            (df1["question_type"] == task)
        ].copy()

        df_sub["gt_norm"] = df_sub["ground_truth"].apply(
            lambda x: normalise_for_sklearn(x, task))
        df_sub["pred_norm"] = df_sub["parsed_answer"].fillna("unknown").apply(
            lambda x: normalise_for_sklearn(x, task))

        # Use union of true labels as display labels
        labels = sorted(df_sub["gt_norm"].unique())

        cm = confusion_matrix(
            df_sub["gt_norm"],
            df_sub["pred_norm"],
            labels=labels
        )

        ax = axes[i][j]
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=labels
        )
        disp.plot(ax=ax, colorbar=False,
                  xticks_rotation=45, cmap="Blues")
        ax.set_title(
            f"{model} | {task.replace('_', ' ').title()}",
            fontsize=10, fontweight="bold"
        )
        ax.set_xlabel("Predicted", fontsize=8)
        ax.set_ylabel("True", fontsize=8)
        ax.tick_params(axis="both", labelsize=7)

plt.tight_layout()
out_path = RESULTS_DIR / "srq1_confusion_matrices.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved to srq1_confusion_matrices.png")

---
## Cell 5 — SRQ1: Per-class accuracy

In [ ]:
print("=" * 65)
print("SRQ1 — Per-Class Accuracy (averaged across all 3 models)")
print("=" * 65)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    "SRQ1 — Per-Class Accuracy (macro avg across models)",
    fontsize=13, fontweight="bold"
)

per_class_records = []

for j, task in enumerate(TASKS):
    df_task = df1[df1["question_type"] == task].copy()

    df_task["gt_norm"] = df_task["ground_truth"].apply(
        lambda x: normalise_for_sklearn(x, task))

    # Average accuracy per class across all models
    class_acc = (
        df_task.groupby("gt_norm")["correct"]
        .mean()
        .sort_values()
    )

    # Store for CSV
    for cls, acc in class_acc.items():
        per_class_records.append({
            "srq": "srq1",
            "task": task,
            "class": cls,
            "accuracy_mean": round(acc, 4)
        })

    print(f"\n  {task}:")
    for cls, acc in class_acc.items():
        bar = "█" * int(acc * 30)
        print(f"    {cls:<20} {acc:.3f}  {bar}")

    # Plot
    ax = axes[j]
    colours = ["#c0392b" if a < 0.5 else "#2980b9" if a >= 0.8
               else "#e67e22" for a in class_acc.values]
    bars = ax.barh(class_acc.index, class_acc.values,
                   color=colours, edgecolor="white")
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Accuracy (avg across models)", fontsize=10)
    ax.set_title(task.replace("_", " ").title(), fontsize=11,
                 fontweight="bold")
    ax.axvline(0.5, color="grey", linestyle="--", alpha=0.5,
               label="0.5 threshold")

    # Value labels
    for bar, val in zip(bars, class_acc.values):
        ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=8)

    # Legend
    patches = [
        mpatches.Patch(color="#c0392b", label="< 0.50"),
        mpatches.Patch(color="#e67e22", label="0.50 – 0.79"),
        mpatches.Patch(color="#2980b9", label="≥ 0.80")
    ]
    ax.legend(handles=patches, fontsize=8, loc="lower right")

plt.tight_layout()
out_path = RESULTS_DIR / "srq1_per_class_accuracy.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()

pd.DataFrame(per_class_records).to_csv(
    RESULTS_DIR / "srq1_per_class_accuracy.csv", index=False)
print(f"\nSaved to srq1_per_class_accuracy.png and .csv")

---
## Cell 6 — SRQ1: Cross-model agreement (Definition A)
For each (movement, task): do all 3 models give the same answer?
Agreement is checked on **normalised** answers so surface differences
(e.g. 'F minor' vs 'f minor') don't count as disagreements.

In [ ]:
print("=" * 65)
print("SRQ1 — Cross-Model Agreement (Definition A)")
print("All-or-nothing: consistent only if all 3 models agree exactly")
print("=" * 65)

agreement_records = []

for task in TASKS:
    df_task = df1[df1["question_type"] == task].copy()
    df_task["answer_norm"] = df_task["parsed_answer"].apply(
        lambda x: normalise_for_sklearn(x, task))

    movements = df_task["piece_id"].unique()

    for mid in movements:
        df_mov = df_task[df_task["piece_id"] == mid]

        model_answers = {}
        for model in MODELS:
            row = df_mov[df_mov["model"] == model]
            if row.empty:
                continue
            if row.iloc[0]["model_response"] == "ERROR":
                continue
            model_answers[model] = row.iloc[0]["answer_norm"]

        if len(model_answers) < 3:
            continue  # skip if any model missing

        answers   = list(model_answers.values())
        all_agree = len(set(answers)) == 1
        majority  = max(set(answers), key=answers.count)

        agreement_records.append({
            "piece_id":      mid,
            "question_type": task,
            "claude_answer": model_answers.get("claude-sonnet-4-6", "missing"),
            "gemini_answer": model_answers.get("gemini-2.5-flash",  "missing"),
            "gpt_answer":    model_answers.get("gpt-5.4",           "missing"),
            "majority":      majority,
            "all_agree":     all_agree
        })

agreement_df = pd.DataFrame(agreement_records)

# ── PRINT SUMMARY ─────────────────────────────────────────────────────────────

print("\nPer-task agreement rate (all 3 models give same answer):")
for task in TASKS:
    sub = agreement_df[agreement_df["question_type"] == task]
    n   = len(sub)
    n_a = sub["all_agree"].sum()
    print(f"  {task:<20}: {n_a}/{n} = {n_a/n:.1%}")

total   = len(agreement_df)
n_agree = agreement_df["all_agree"].sum()
print(f"\nCombined (both tasks): {n_agree}/{total} = {n_agree/total:.1%}")
print("  (for reference only — tasks differ in difficulty)")

print("\nPer-model majority match rate:")
for model, col in [
    ("claude-sonnet-4-6", "claude_answer"),
    ("gemini-2.5-flash",  "gemini_answer"),
    ("gpt-5.4",           "gpt_answer")
]:
    n_match = agreement_df.apply(
        lambda row: row[col] == row["majority"], axis=1
    ).sum()
    print(f"  {model:<25}: {n_match}/{total} = {n_match/total:.1%}")

# ── DISAGREEMENT CASES ────────────────────────────────────────────────────────

disagreements = agreement_df[~agreement_df["all_agree"]]
print(f"\nDisagreement cases: {len(disagreements)} / {total}")
if len(disagreements) > 0:
    print("\nFirst 15 disagreements:")
    print(
        disagreements[
            ["piece_id", "question_type",
             "claude_answer", "gemini_answer", "gpt_answer"]
        ].head(15).to_string(index=False)
    )

# ── SAVE ─────────────────────────────────────────────────────────────────────
agreement_df.to_csv(RESULTS_DIR / "srq1_agreement.csv", index=False)
print(f"\nFull agreement table saved to srq1_agreement.csv")

---
## Cell 6b — Pairwise Cohen's Kappa Between Models
Extends the cross-model agreement analysis by computing agreement
corrected for chance (Cohen's κ). Also defines `ACCIDENTAL_ORDER`
used by the sorted confusion matrices below.

In [ ]:
from sklearn.metrics import cohen_kappa_score

print('=' * 65)
print('ANALYSIS 7 — Pairwise Cohen\'s Kappa Between Models')
print('Treating each model as an independent annotator.')
print('κ measures agreement corrected for chance.')
print('=' * 65)

# ── KEY SIGNATURE ORDER (sorted by number of sharps/flats) ───────────────────
# Used here AND by the sorted confusion matrix cells below.
ACCIDENTAL_ORDER = [
    # Flats (most to least)
    'gb major', 'eb minor',   # 6 flats
    'db major', 'bb minor',   # 5 flats
    'ab major', 'f minor',    # 4 flats
    'eb major', 'c minor',    # 3 flats
    'bb major', 'g minor',    # 2 flats
    'f major',  'd minor',    # 1 flat
    # Natural
    'c major',  'a minor',    # 0
    # Sharps (least to most)
    'g major',  'e minor',    # 1 sharp
    'd major',  'b minor',    # 2 sharps
    'a major',  'f# minor',   # 3 sharps
    'e major',  'c# minor',   # 4 sharps
    'b major',  'g# minor',   # 5 sharps
    'f# major', 'd# minor',   # 6 sharps
]

TIME_ORDER = ['2/2', '2/4', '3/4', '3/8', '4/4',
              '6/8', '9/8', '9/16', '12/8']

MODEL_PAIRS = [
    ('claude-sonnet-4-6', 'gemini-2.5-flash', 'Claude vs Gemini'),
    ('claude-sonnet-4-6', 'gpt-5.4',          'Claude vs GPT'),
    ('gemini-2.5-flash',  'gpt-5.4',          'Gemini vs GPT'),
]

kappa_records = []

def compute_pairwise_kappa(df, srq_label, fmt=None):
    print(f"\n  {'─'*55}")
    if fmt:
        print(f'  {srq_label} | Format: {fmt.upper()}')
        df = df[df['input_type'] == fmt]
    else:
        print(f'  {srq_label}')
    print(f"  {'─'*55}")

    for task in TASKS:
        print(f'\n    Task: {task}')
        df_task = df[
            (df['question_type'] == task) &
            (df['model_response'] != 'ERROR')
        ].copy()
        df_task['pred_norm'] = df_task['parsed_answer'].fillna('unknown').apply(
            lambda x: normalise_for_sklearn(x, task))

        for model_a, model_b, pair_label in MODEL_PAIRS:
            df_a = df_task[df_task['model'] == model_a].set_index('piece_id')['pred_norm']
            df_b = df_task[df_task['model'] == model_b].set_index('piece_id')['pred_norm']
            shared = df_a.index.intersection(df_b.index)
            if len(shared) < 2:
                print(f'      {pair_label}: not enough shared movements')
                continue
            try:
                kappa = cohen_kappa_score(
                    df_a.loc[shared].values,
                    df_b.loc[shared].values
                )
            except Exception as e:
                print(f'      {pair_label}: error — {e}')
                kappa = float('nan')
            interp = ('Almost perfect' if kappa > 0.80 else
                      'Substantial'    if kappa > 0.60 else
                      'Moderate'       if kappa > 0.40 else
                      'Fair'           if kappa > 0.20 else
                      'Slight'         if kappa > 0.00 else
                      'Poor / negative')
            print(f'      {pair_label:<25}: κ = {kappa:>6.3f}  [{interp}]')
            kappa_records.append({
                'srq': srq_label, 'format': fmt if fmt else 'pdf',
                'task': task, 'pair': pair_label,
                'model_a': model_a, 'model_b': model_b,
                'n_shared': len(shared),
                'kappa': round(kappa, 4),
                'interpretation': interp
            })

# Run
compute_pairwise_kappa(df1, 'SRQ1')
compute_pairwise_kappa(df3, 'SRQ3', fmt='pdf')
compute_pairwise_kappa(df3, 'SRQ3', fmt='png')

# ── HEATMAP ───────────────────────────────────────────────────────────────────
kappa_df = pd.DataFrame(kappa_records)

fig, axes = plt.subplots(len(TASKS), 3, figsize=(16, 5 * len(TASKS)))
fig.suptitle(
    'Analysis 7 — Pairwise Cohen\'s Kappa Between Models\n'
    '(rows = tasks, columns = SRQ1 PDF / SRQ3 PDF / SRQ3 PNG)',
    fontsize=13, fontweight='bold', y=1.01
)

conditions = [
    ('SRQ1', 'pdf', 'SRQ1 (PDF)'),
    ('SRQ3', 'pdf', 'SRQ3 (PDF)'),
    ('SRQ3', 'png', 'SRQ3 (PNG)'),
]
model_short = {
    'claude-sonnet-4-6': 'Claude',
    'gemini-2.5-flash':  'Gemini',
    'gpt-5.4':           'GPT'
}
short = ['Claude', 'Gemini', 'GPT']

for row, task in enumerate(TASKS):
    for col, (srq, fmt, label) in enumerate(conditions):
        ax = axes[row][col]
        sub = kappa_df[
            (kappa_df['srq'] == srq) &
            (kappa_df['format'] == fmt) &
            (kappa_df['task'] == task)
        ]
        mat = np.ones((3, 3))
        for _, r in sub.iterrows():
            i = short.index(model_short[r['model_a']])
            j = short.index(model_short[r['model_b']])
            mat[i][j] = r['kappa']
            mat[j][i] = r['kappa']
        im = ax.imshow(mat, cmap='RdYlGn', vmin=-0.2, vmax=1.0, aspect='auto')
        ax.set_xticks(range(3))
        ax.set_yticks(range(3))
        ax.set_xticklabels(short, fontsize=10)
        ax.set_yticklabels(short, fontsize=10)
        ax.set_title(
            f'{label}\n{task.replace("_", " ").title()}',
            fontsize=10, fontweight='bold'
        )
        for i in range(3):
            for j in range(3):
                val = mat[i][j]
                colour = 'white' if val < 0.3 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=11, fontweight='bold', color=colour)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'deep_7_kappa_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()
kappa_df.to_csv(RESULTS_DIR / 'deep_7_kappa.csv', index=False)
print('\nSaved to deep_7_kappa_heatmap.png and deep_7_kappa.csv')
print('\nInterpretation guide:')
print('  κ > 0.80  Almost perfect   κ 0.61-0.80  Substantial')
print('  κ 0.41-0.60  Moderate      κ 0.21-0.40  Fair')
print('  κ 0.00-0.20  Slight        κ < 0.00     Poor / negative')


---
## Cell 6c — Confusion Matrices Sorted by Accidental Count
Replaces the alphabetically-sorted confusion matrices (Cells 4 and 10).
- Labels run: 6 flats → 0 accidentals → 6 sharps.
- Red lines divide flat / natural / sharp zones.
- Errors near the diagonal = adjacent-key mistakes (systematic).
- Errors crossing zone lines = accidental direction confused.

---
- Errors near the diagonal mean the model predicted a key with a similar number of accidentals — e.g. predicting D major (2 sharps) when the truth is A major (3 sharps). That's a systematic, musically structured error
- Errors far from the diagonal — e.g. predicting a flat key when the truth is a sharp key — mean the model is making unstructured guesses
- Errors within one zone (all within the red lines) mean the model got the accidental direction right but miscounted
- Errors crossing the red zone lines mean the model confused sharps with flats entirely — connects directly to your Analysis 3 results

In [ ]:
def get_sorted_labels(df_subset, task):
    present = df_subset['gt_norm'].unique().tolist()
    if task == 'key_signature':
        ordered  = [k for k in ACCIDENTAL_ORDER if k in present]
        ordered += [k for k in present if k not in ordered]
    else:
        ordered  = [t for t in TIME_ORDER if t in present]
        ordered += [t for t in present if t not in ordered]
    return ordered


def plot_sorted_confusion(df, srq_label, fmt_list):
    for task in TASKS:
        n_cols  = len(fmt_list)
        fig, axes = plt.subplots(
            len(MODELS), n_cols,
            figsize=(10 * n_cols, 7 * len(MODELS))
        )
        if len(MODELS) == 1: axes = [axes]
        if n_cols == 1:      axes = [[ax] for ax in axes]

        fig.suptitle(
            f'{srq_label} — Confusion Matrix | '
            f'{task.replace("_", " ").title()}\n'
            f'Sorted by accidentals (left = most flats, right = most sharps)',
            fontsize=12, fontweight='bold', y=1.01
        )

        # Shared sorted labels across all conditions
        df_all = df[df['question_type'] == task].copy()
        df_all['gt_norm']   = df_all['ground_truth'].apply(lambda x: normalise_for_sklearn(x, task))
        df_all['pred_norm'] = df_all['parsed_answer'].fillna('unknown').apply(lambda x: normalise_for_sklearn(x, task))
        labels = get_sorted_labels(df_all, task)

        for i, model in enumerate(MODELS):
            for j, fmt in enumerate(fmt_list):
                ax = axes[i][j]

                if 'input_type' in df.columns:
                    df_sub = df[
                        (df['model'] == model) &
                        (df['question_type'] == task) &
                        (df['input_type'] == fmt)
                    ].copy()
                else:
                    df_sub = df[
                        (df['model'] == model) &
                        (df['question_type'] == task)
                    ].copy()

                df_sub['gt_norm']   = df_sub['ground_truth'].apply(lambda x: normalise_for_sklearn(x, task))
                df_sub['pred_norm'] = df_sub['parsed_answer'].fillna('unknown').apply(lambda x: normalise_for_sklearn(x, task))

                cm = confusion_matrix(df_sub['gt_norm'], df_sub['pred_norm'], labels=labels)

                ax.imshow(cm, cmap='Blues', aspect='auto')
                ax.set_xticks(range(len(labels)))
                ax.set_yticks(range(len(labels)))
                ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=7)
                ax.set_yticklabels(labels, fontsize=7)
                ax.set_xlabel('Predicted', fontsize=9)
                ax.set_ylabel('True', fontsize=9)

                short_m = (model.replace('claude-sonnet-4-6','Claude')
                               .replace('gemini-2.5-flash','Gemini')
                               .replace('gpt-5.4','GPT'))
                ax.set_title(f'{short_m} | {fmt.upper()}', fontsize=10, fontweight='bold')

                for r in range(len(labels)):
                    for c in range(len(labels)):
                        val = cm[r, c]
                        if val > 0:
                            colour = 'white' if val > cm.max() * 0.6 else 'black'
                            ax.text(c, r, str(val), ha='center', va='center',
                                    fontsize=6, color=colour)

                # Zone dividers for key signatures only
                if task == 'key_signature':
                    flat_labels    = [l for l in labels if l in ACCIDENTAL_ORDER[:12]]
                    natural_labels = [l for l in labels if l in ACCIDENTAL_ORDER[12:14]]
                    sharp_labels   = [l for l in labels if l in ACCIDENTAL_ORDER[14:]]
                    n_flat = len(flat_labels)
                    n_nat  = len(natural_labels)
                    if n_flat > 0:
                        ax.axhline(n_flat - 0.5, color='red',    linewidth=1.2, alpha=0.6)
                        ax.axvline(n_flat - 0.5, color='red',    linewidth=1.2, alpha=0.6)
                    if n_flat + n_nat > 0 and n_flat > 0:
                        ax.axhline(n_flat + n_nat - 0.5, color='orange', linewidth=1.2, alpha=0.6)
                        ax.axvline(n_flat + n_nat - 0.5, color='orange', linewidth=1.2, alpha=0.6)
                    if n_flat > 0:
                        ax.text(-0.8, n_flat / 2 - 0.5, chr(9837),
                                ha='center', va='center', fontsize=14, color='red', alpha=0.7)
                    if n_nat > 0:
                        ax.text(-0.8, n_flat + n_nat / 2 - 0.5, chr(9838),
                                ha='center', va='center', fontsize=14, color='orange', alpha=0.7)
                    if len(sharp_labels) > 0:
                        ax.text(-0.8, n_flat + n_nat + len(sharp_labels) / 2 - 0.5,
                                chr(9839), ha='center', va='center',
                                fontsize=14, color='blue', alpha=0.7)

        plt.tight_layout()
        safe_srq = srq_label.replace(' ', '_')
        out_path = RESULTS_DIR / f'sorted_cm_{safe_srq}_{task}.png'
        plt.savefig(out_path, dpi=130, bbox_inches='tight')
        plt.show()
        print(f'Saved to sorted_cm_{safe_srq}_{task}.png')


# SRQ1 — add dummy input_type if missing
df1_plot = df1.copy()
if 'input_type' not in df1_plot.columns:
    df1_plot['input_type'] = 'pdf'

plot_sorted_confusion(df1_plot, 'SRQ1', ['pdf'])
plot_sorted_confusion(df3,      'SRQ3', ['pdf', 'png'])

print('\nAll sorted confusion matrices saved.')
print('\nWhat to look for:')
print('  Errors near diagonal     = adjacent-key errors (systematic)')
print('  Errors far from diagonal = random/unstructured errors')
print('  Errors within one zone   = direction preserved, count wrong')
print('  Errors crossing zones    = accidental direction confused')


### ANALYSIS — SRQ1: Claude vs GPT Agreement Breakdown

In [ ]:
print("=" * 65)
print("ANALYSIS — SRQ1: Claude vs GPT Agreement Breakdown")
print("Excluding Gemini (too few correct outputs to be informative)")
print("2C  = both correct")
print("2W  = both wrong")
print("1C1W = Claude correct, GPT wrong")
print("1W1C = Claude wrong, GPT correct")
print("=" * 65)

FOCUS_MODELS = ["claude-sonnet-4-6", "gpt-5.4"]

agreement_records = []

for task in TASKS:
    df_task = df1[
        (df1["question_type"] == task) &
        (df1["model_response"] != "ERROR")
    ]

    movements = df_task["piece_id"].unique()

    task_2c   = []
    task_2w   = []
    task_1c1w = []
    task_1w1c = []

    for mid in movements:
        df_mov = df_task[df_task["piece_id"] == mid]

        # Get correctness for Claude and GPT only
        results = {}
        for model in FOCUS_MODELS:
            row = df_mov[df_mov["model"] == model]
            if not row.empty:
                results[model] = bool(row.iloc[0]["correct"])

        # Skip if either model missing
        if len(results) < 2:
            continue

        claude_correct = results["claude-sonnet-4-6"]
        gpt_correct    = results["gpt-5.4"]

        if claude_correct and gpt_correct:
            category = "2C"
            task_2c.append(mid)
        elif not claude_correct and not gpt_correct:
            category = "2W"
            task_2w.append(mid)
        elif claude_correct and not gpt_correct:
            category = "1C1W"
            task_1c1w.append(mid)
        else:
            category = "1W1C"
            task_1w1c.append(mid)

        agreement_records.append({
            "task":           task,
            "piece_id":       mid,
            "claude_correct": claude_correct,
            "gpt_correct":    gpt_correct,
            "category":       category
        })

    total = len(task_2c) + len(task_2w) + len(task_1c1w) + len(task_1w1c)

    print(f"\n  Task: {task}  (n={total} movements)")
    print(f"  {'─'*50}")
    print(f"  2C   (both correct):         "
          f"{len(task_2c):>3} / {total} = {len(task_2c)/total:.1%}")
    print(f"  2W   (both wrong):           "
          f"{len(task_2w):>3} / {total} = {len(task_2w)/total:.1%}")
    print(f"  1C1W (Claude only correct):  "
          f"{len(task_1c1w):>3} / {total} = {len(task_1c1w)/total:.1%}")
    print(f"  1W1C (GPT only correct):     "
          f"{len(task_1w1c):>3} / {total} = {len(task_1w1c)/total:.1%}")

    if task_2w:
        print(f"\n  2W movements (both fail — genuinely hard):")
        print(f"    {sorted(task_2w)}")
    if task_1c1w:
        print(f"\n  1C1W movements (Claude edge over GPT):")
        print(f"    {sorted(task_1c1w)}")
    if task_1w1c:
        print(f"\n  1W1C movements (GPT edge over Claude):")
        print(f"    {sorted(task_1w1c)}")

# ── PLOT ──────────────────────────────────────────────────────────────────────

agreement_df = pd.DataFrame(agreement_records)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "SRQ1 — Claude vs GPT Agreement Breakdown\n"
    "(Gemini excluded — too few correct outputs)",
    fontsize=12, fontweight="bold"
)

cat_colours = {
    "2C":   "#4a7c8e",   # teal
    "2W":   "#c05c7e",   # rose
    "1C1W": "#7e6eab",   # purple
    "1W1C": "#e08c5a",   # warm orange
}
cat_labels = {
    "2C":   "2C — Both correct",
    "2W":   "2W — Both wrong",
    "1C1W": "1C1W — Claude only correct",
    "1W1C": "1W1C — GPT only correct",
}
cat_order = ["2C", "2W", "1C1W", "1W1C"]

for ax, task in zip(axes, TASKS):
    sub    = agreement_df[agreement_df["task"] == task]
    counts = sub["category"].value_counts()
    total  = len(sub)

    values = [counts.get(c, 0) for c in cat_order]
    pcts   = [v / total * 100 for v in values]
    colours = [cat_colours[c] for c in cat_order]
    labels  = [f"{cat_labels[c]}\n({v}, {p:.1f}%)"
               for c, v, p in zip(cat_order, values, pcts)]

    wedges, texts = ax.pie(
        values,
        colors=colours,
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 2}
    )
    ax.legend(wedges, labels, loc="lower center",
              bbox_to_anchor=(0.5, -0.35),
              fontsize=8, frameon=False)
    ax.set_title(
        task.replace("_", " ").title(),
        fontsize=11, fontweight="bold"
    )

plt.tight_layout()
out_path = RESULTS_DIR / "srq1_claude_gpt_agreement.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()

agreement_df.to_csv(RESULTS_DIR / "srq1_claude_gpt_agreement.csv", index=False)
print(f"\nSaved to srq1_claude_gpt_agreement.png and .csv")
print("\nNote: 2W movements overlap with Analysis 5 hard movements.")
print("1C1W and 1W1C movements reveal model-specific strengths.")

---
# SRQ2 Analysis

## Accuracy summary table

In [ ]:
print("=" * 65)
print("SRQ2 — Accuracy Summary (CoT, PDF)")
print("=" * 65)
print(f"{'model':<25} {'task':<20} {'n':>4} {'errors':>6} "
      f"{'accuracy':>9} {'precision':>10} {'recall':>7} {'f1':>6}")
print("-" * 90)

srq2_summary = []

for model in MODELS:
    for task in TASKS:
        sub = df2[(df2["model"] == model) & (df2["question_type"] == task)]
        m   = compute_metrics(sub, task)
        print(f"{model:<25} {task:<20} {m['n']:>4} {m['n_errors']:>6} "
              f"{m['accuracy']:>9.3f} {m['precision']:>10.3f} "
              f"{m['recall']:>7.3f} {m['f1']:>6.3f}")
        srq2_summary.append({"model": model, "task": task, **m})

print("\nAll metrics use macro averaging.")

srq2_summary_df = pd.DataFrame(srq2_summary)
srq2_summary_df.to_csv(RESULTS_DIR / "srq2_summary.csv", index=False)
print(f"Saved to srq2_summary.csv")

## Per-class accuracy

In [ ]:
print("=" * 65)
print("SRQ2 — Per-Class Accuracy (averaged across all 3 models)")
print("=" * 65)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    "SRQ2 — Per-Class Accuracy (macro avg across models)",
    fontsize=13, fontweight="bold"
)

per_class_records_srq2 = []

for j, task in enumerate(TASKS):
    df_task = df2[df2["question_type"] == task].copy()

    df_task["gt_norm"] = df_task["ground_truth"].apply(
        lambda x: normalise_for_sklearn(x, task))

    class_acc = (
        df_task.groupby("gt_norm")["correct"]
        .mean()
        .sort_values()
    )

    for cls, acc in class_acc.items():
        per_class_records_srq2.append({
            "srq":           "srq2",
            "task":          task,
            "class":         cls,
            "accuracy_mean": round(acc, 4)
        })

    print(f"\n  {task}:")
    for cls, acc in class_acc.items():
        bar = "█" * int(acc * 30)
        print(f"    {cls:<20} {acc:.3f}  {bar}")

    ax = axes[j]
    colours = ["#c0392b" if a < 0.5 else "#2980b9" if a >= 0.8
               else "#e67e22" for a in class_acc.values]
    bars = ax.barh(class_acc.index, class_acc.values,
                   color=colours, edgecolor="white")
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Accuracy (avg across models)", fontsize=10)
    ax.set_title(task.replace("_", " ").title(), fontsize=11,
                 fontweight="bold")
    ax.axvline(0.5, color="grey", linestyle="--", alpha=0.5,
               label="0.5 threshold")

    for bar, val in zip(bars, class_acc.values):
        ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=8)

    patches = [
        mpatches.Patch(color="#c0392b", label="< 0.50"),
        mpatches.Patch(color="#e67e22", label="0.50 – 0.79"),
        mpatches.Patch(color="#2980b9", label="≥ 0.80")
    ]
    ax.legend(handles=patches, fontsize=8, loc="lower right")

plt.tight_layout()
out_path = RESULTS_DIR / "srq2_per_class_accuracy.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()

pd.DataFrame(per_class_records_srq2).to_csv(
    RESULTS_DIR / "srq2_per_class_accuracy.csv", index=False)
print(f"\nSaved to srq2_per_class_accuracy.png and .csv")

## Cross-model agreement (all 3 models)

In [ ]:
print("=" * 65)
print("SRQ2 — Cross-Model Agreement (Definition A)")
print("All-or-nothing: consistent only if all 3 models agree exactly")
print("=" * 65)

agreement_records_srq2 = []

for task in TASKS:
    df_task = df2[df2["question_type"] == task].copy()
    df_task["answer_norm"] = df_task["parsed_answer"].apply(
        lambda x: normalise_for_sklearn(x, task))

    movements = df_task["piece_id"].unique()

    for mid in movements:
        df_mov = df_task[df_task["piece_id"] == mid]

        model_answers = {}
        for model in MODELS:
            row = df_mov[df_mov["model"] == model]
            if row.empty:
                continue
            if row.iloc[0]["model_response"] == "ERROR":
                continue
            model_answers[model] = row.iloc[0]["answer_norm"]

        if len(model_answers) < 3:
            continue

        answers   = list(model_answers.values())
        all_agree = len(set(answers)) == 1
        majority  = max(set(answers), key=answers.count)

        agreement_records_srq2.append({
            "piece_id":      mid,
            "question_type": task,
            "claude_answer": model_answers.get("claude-sonnet-4-6", "missing"),
            "gemini_answer": model_answers.get("gemini-2.5-flash",  "missing"),
            "gpt_answer":    model_answers.get("gpt-5.4",           "missing"),
            "majority":      majority,
            "all_agree":     all_agree
        })

agreement_df_srq2 = pd.DataFrame(agreement_records_srq2)

print("\nPer-task agreement rate (all 3 models give same answer):")
for task in TASKS:
    sub = agreement_df_srq2[agreement_df_srq2["question_type"] == task]
    n   = len(sub)
    n_a = sub["all_agree"].sum()
    print(f"  {task:<20}: {n_a}/{n} = {n_a/n:.1%}")

total   = len(agreement_df_srq2)
n_agree = agreement_df_srq2["all_agree"].sum()
print(f"\nCombined (both tasks): {n_agree}/{total} = {n_agree/total:.1%}")
print("  (for reference only — tasks differ in difficulty)")

print("\nPer-model majority match rate:")
for model, col in [
    ("claude-sonnet-4-6", "claude_answer"),
    ("gemini-2.5-flash",  "gemini_answer"),
    ("gpt-5.4",           "gpt_answer")
]:
    n_match = agreement_df_srq2.apply(
        lambda row: row[col] == row["majority"], axis=1
    ).sum()
    print(f"  {model:<25}: {n_match}/{total} = {n_match/total:.1%}")

disagreements = agreement_df_srq2[~agreement_df_srq2["all_agree"]]
print(f"\nDisagreement cases: {len(disagreements)} / {total}")
if len(disagreements) > 0:
    print("\nFirst 15 disagreements:")
    print(
        disagreements[
            ["piece_id", "question_type",
             "claude_answer", "gemini_answer", "gpt_answer"]
        ].head(15).to_string(index=False)
    )

agreement_df_srq2.to_csv(RESULTS_DIR / "srq2_agreement.csv", index=False)
print(f"\nFull agreement table saved to srq2_agreement.csv")

## Claude vs GPT agreement (Gemini excluded)

In [ ]:
print("=" * 65)
print("ANALYSIS — SRQ2: Claude vs GPT Agreement Breakdown")
print("Excluding Gemini (too few correct outputs to be informative)")
print("2C  = both correct")
print("2W  = both wrong")
print("1C1W = Claude correct, GPT wrong")
print("1W1C = Claude wrong, GPT correct")
print("=" * 65)

FOCUS_MODELS = ["claude-sonnet-4-6", "gpt-5.4"]

agreement_records_cg_srq2 = []

for task in TASKS:
    df_task = df2[
        (df2["question_type"] == task) &
        (df2["model_response"] != "ERROR")
    ]

    movements = df_task["piece_id"].unique()

    task_2c   = []
    task_2w   = []
    task_1c1w = []
    task_1w1c = []

    for mid in movements:
        df_mov = df_task[df_task["piece_id"] == mid]

        results = {}
        for model in FOCUS_MODELS:
            row = df_mov[df_mov["model"] == model]
            if not row.empty:
                results[model] = bool(row.iloc[0]["correct"])

        if len(results) < 2:
            continue

        claude_correct = results["claude-sonnet-4-6"]
        gpt_correct    = results["gpt-5.4"]

        if claude_correct and gpt_correct:
            category = "2C"
            task_2c.append(mid)
        elif not claude_correct and not gpt_correct:
            category = "2W"
            task_2w.append(mid)
        elif claude_correct and not gpt_correct:
            category = "1C1W"
            task_1c1w.append(mid)
        else:
            category = "1W1C"
            task_1w1c.append(mid)

        agreement_records_cg_srq2.append({
            "task":           task,
            "piece_id":       mid,
            "claude_correct": claude_correct,
            "gpt_correct":    gpt_correct,
            "category":       category
        })

    total = len(task_2c) + len(task_2w) + len(task_1c1w) + len(task_1w1c)

    print(f"\n  Task: {task}  (n={total} movements)")
    print(f"  {'─'*50}")
    print(f"  2C   (both correct):         "
          f"{len(task_2c):>3} / {total} = {len(task_2c)/total:.1%}")
    print(f"  2W   (both wrong):           "
          f"{len(task_2w):>3} / {total} = {len(task_2w)/total:.1%}")
    print(f"  1C1W (Claude only correct):  "
          f"{len(task_1c1w):>3} / {total} = {len(task_1c1w)/total:.1%}")
    print(f"  1W1C (GPT only correct):     "
          f"{len(task_1w1c):>3} / {total} = {len(task_1w1c)/total:.1%}")

    if task_2w:
        print(f"\n  2W movements (both fail — genuinely hard):")
        print(f"    {sorted(task_2w)}")
    if task_1c1w:
        print(f"\n  1C1W movements (Claude edge over GPT):")
        print(f"    {sorted(task_1c1w)}")
    if task_1w1c:
        print(f"\n  1W1C movements (GPT edge over Claude):")
        print(f"    {sorted(task_1w1c)}")

# ── PLOT ──────────────────────────────────────────────────────────────────────

agreement_df_cg_srq2 = pd.DataFrame(agreement_records_cg_srq2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "SRQ2 — Claude vs GPT Agreement Breakdown\n"
    "(Gemini excluded — too few correct outputs)",
    fontsize=12, fontweight="bold"
)

cat_colours = {
    "2C":   "#4a7c8e",
    "2W":   "#c05c7e",
    "1C1W": "#7e6eab",
    "1W1C": "#e08c5a",
}
cat_labels = {
    "2C":   "2C — Both correct",
    "2W":   "2W — Both wrong",
    "1C1W": "1C1W — Claude only correct",
    "1W1C": "1W1C — GPT only correct",
}
cat_order = ["2C", "2W", "1C1W", "1W1C"]

for ax, task in zip(axes, TASKS):
    sub    = agreement_df_cg_srq2[agreement_df_cg_srq2["task"] == task]
    counts = sub["category"].value_counts()
    total  = len(sub)

    values  = [counts.get(c, 0) for c in cat_order]
    pcts    = [v / total * 100 for v in values]
    colours = [cat_colours[c] for c in cat_order]
    labels  = [f"{cat_labels[c]}\n({v}, {p:.1f}%)"
               for c, v, p in zip(cat_order, values, pcts)]

    wedges, texts = ax.pie(
        values,
        colors=colours,
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 2}
    )
    ax.legend(wedges, labels, loc="lower center",
              bbox_to_anchor=(0.5, -0.35),
              fontsize=8, frameon=False)
    ax.set_title(
        task.replace("_", " ").title(),
        fontsize=11, fontweight="bold"
    )

plt.tight_layout()
out_path = RESULTS_DIR / "srq2_claude_gpt_agreement.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()

agreement_df_cg_srq2.to_csv(RESULTS_DIR / "srq2_claude_gpt_agreement.csv", index=False)
print(f"\nSaved to srq2_claude_gpt_agreement.png and .csv")
print("\nNote: 2W movements overlap with Analysis 5 hard movements.")
print("1C1W and 1W1C movements reveal model-specific strengths.")

---
# SRQ3 Analysis
## Cell 7 — SRQ3: Accuracy summary table

In [ ]:
print("=" * 75)
print("SRQ3 — Accuracy Summary (Role prompt, PDF vs PNG)")
print("=" * 75)
print(f"{'model':<25} {'format':<6} {'task':<20} {'n':>4} {'errors':>6} "
      f"{'accuracy':>9} {'precision':>10} {'recall':>7} {'f1':>6}")
print("-" * 95)

srq3_summary = []

for model in MODELS:
    for fmt in ["pdf", "png"]:
        for task in TASKS:
            sub = df3[
                (df3["model"] == model) &
                (df3["input_type"] == fmt) &
                (df3["question_type"] == task)
            ]
            m = compute_metrics(sub, task)
            print(f"{model:<25} {fmt:<6} {task:<20} {m['n']:>4} "
                  f"{m['n_errors']:>6} {m['accuracy']:>9.3f} "
                  f"{m['precision']:>10.3f} {m['recall']:>7.3f} "
                  f"{m['f1']:>6.3f}")
            srq3_summary.append({
                "model": model, "format": fmt, "task": task, **m})

print("\nAll metrics use macro averaging.")

srq3_summary_df = pd.DataFrame(srq3_summary)
srq3_summary_df.to_csv(RESULTS_DIR / "srq3_summary.csv", index=False)
print(f"Saved to srq3_summary.csv")

---
## Cell 8 — SRQ3: McNemar's test (PDF vs PNG)
**Why McNemar's?** The same movements appear in both PDF and PNG conditions,
so the data is paired. McNemar's is the standard test for paired binary
outcomes (correct / incorrect). It tests whether the disagreements between
the two formats are symmetric — i.e. does one format produce significantly
more correct answers than the other?

H0: no significant difference in accuracy between PDF and PNG.

In [ ]:
print("=" * 65)
print("SRQ3 — McNemar's Test: PDF vs PNG")
print("H0: no significant difference in accuracy between formats")
print("=" * 65)

mcnemar_records = []

for model in MODELS:
    for task in TASKS:
        df_pdf = df3[
            (df3["model"] == model) &
            (df3["question_type"] == task) &
            (df3["input_type"] == "pdf")
        ].set_index("piece_id")["correct"]

        df_png = df3[
            (df3["model"] == model) &
            (df3["question_type"] == task) &
            (df3["input_type"] == "png")
        ].set_index("piece_id")["correct"]

        # Align on shared movements only
        shared      = df_pdf.index.intersection(df_png.index)
        pdf_correct = df_pdf.loc[shared].astype(bool)
        png_correct = df_png.loc[shared].astype(bool)

        # 2x2 contingency table
        both_correct = (pdf_correct & png_correct).sum()
        pdf_only     = (pdf_correct & ~png_correct).sum()  # PDF right PNG wrong
        png_only     = (~pdf_correct & png_correct).sum()  # PNG right PDF wrong
        both_wrong   = (~pdf_correct & ~png_correct).sum()

        table = [[both_correct, pdf_only],
                 [png_only,     both_wrong]]

        # McNemar test — exact=True recommended when off-diagonal counts are small
        try:
            result  = mcnemar(table, exact=True, correction=False)
            p_value = result.pvalue
        except Exception:
            p_value = float("nan")

        sig = ("***" if p_value < 0.001 else
               "**"  if p_value < 0.01  else
               "*"   if p_value < 0.05  else "ns")

        acc_pdf = pdf_correct.mean()
        acc_png = png_correct.mean()
        delta   = acc_png - acc_pdf  # positive = PNG better

        print(f"\n  {model} | {task}")
        print(f"    n (shared movements):  {len(shared)}")
        print(f"    PDF acc = {acc_pdf:.3f}   PNG acc = {acc_png:.3f}   "
              f"Δ = {delta:+.3f} ({'PNG better' if delta > 0 else 'PDF better'})")
        print(f"    Contingency table:")
        print(f"      Both correct:  {both_correct}")
        print(f"      PDF only:      {pdf_only}")
        print(f"      PNG only:      {png_only}")
        print(f"      Both wrong:    {both_wrong}")
        print(f"    McNemar p = {p_value:.4f}  {sig}")

        mcnemar_records.append({
            "model":        model,
            "task":         task,
            "n_shared":     int(len(shared)),
            "acc_pdf":      round(acc_pdf, 4),
            "acc_png":      round(acc_png, 4),
            "delta":        round(delta, 4),
            "both_correct": int(both_correct),
            "pdf_only":     int(pdf_only),
            "png_only":     int(png_only),
            "both_wrong":   int(both_wrong),
            "p_value":      round(p_value, 4),
            "significant":  sig
        })

print("\n" + "-" * 65)
print("Significance: *** p<.001  ** p<.01  * p<.05  ns = not significant")

mcnemar_df = pd.DataFrame(mcnemar_records)
mcnemar_df.to_csv(RESULTS_DIR / "srq3_mcnemar.csv", index=False)
print(f"Saved to srq3_mcnemar.csv")

---
## Cell 9 — SRQ3: Format effect direction
When PDF and PNG produced different answers for the same movement,
which format was correct? Breaks down into three cases:
- **PDF better**: PDF correct, PNG wrong
- **PNG better**: PNG correct, PDF wrong
- **Both wrong**: both wrong but gave different answers

In [ ]:
print("=" * 65)
print("SRQ3 — Format Effect Direction")
print("When PDF and PNG disagreed, which format was correct?")
print("=" * 65)

direction_records = []

for model in MODELS:
    print(f"\n  {model}")
    for task in TASKS:
        df_pdf = df3[
            (df3["model"] == model) &
            (df3["question_type"] == task) &
            (df3["input_type"] == "pdf")
        ].set_index("piece_id")

        df_png = df3[
            (df3["model"] == model) &
            (df3["question_type"] == task) &
            (df3["input_type"] == "png")
        ].set_index("piece_id")

        shared  = df_pdf.index.intersection(df_png.index)
        pdf_c   = df_pdf.loc[shared, "correct"].astype(bool)
        png_c   = df_png.loc[shared, "correct"].astype(bool)

        # Only look at cases where answers disagreed
        pdf_ans = df_pdf.loc[shared, "parsed_answer"].fillna("")
        png_ans = df_png.loc[shared, "parsed_answer"].fillna("")

        # Normalise before comparing answers
        pdf_norm = pdf_ans.apply(lambda x: normalise_for_sklearn(x, task))
        png_norm = png_ans.apply(lambda x: normalise_for_sklearn(x, task))
        answer_differs = pdf_norm != png_norm

        pdf_better  = (pdf_c & ~png_c).sum()
        png_better  = (~pdf_c & png_c).sum()
        both_differ_wrong = (answer_differs & ~pdf_c & ~png_c).sum()
        n_disagree  = answer_differs.sum()

        print(f"    {task:<20}: {n_disagree} disagreements  "
              f"→  PDF better={pdf_better}, "
              f"PNG better={png_better}, "
              f"both wrong={both_differ_wrong}")

        direction_records.append({
            "model":            model,
            "task":             task,
            "n_disagreements":  int(n_disagree),
            "pdf_better":       int(pdf_better),
            "png_better":       int(png_better),
            "both_wrong":       int(both_differ_wrong)
        })

direction_df = pd.DataFrame(direction_records)
direction_df.to_csv(RESULTS_DIR / "srq3_format_direction.csv", index=False)
print(f"\nSaved to srq3_format_direction.csv")

---
## Cell 10 — SRQ3: Confusion matrices (per model, per task, per format)

In [ ]:
print("=" * 65)
print("SRQ3 — Confusion Matrices (PDF vs PNG, key signature only)")
print("=" * 65)

FORMATS = ["pdf", "png"]

# One figure per task to keep it readable
for task in TASKS:
    fig, axes = plt.subplots(
        len(MODELS), len(FORMATS),
        figsize=(16, 7 * len(MODELS))
    )
    fig.suptitle(
        f"SRQ3 — Confusion Matrices | {task.replace('_',' ').title()}",
        fontsize=13, fontweight="bold", y=1.01
    )

    # Determine a shared label set across all model/format combos for this task
    all_gt = df3[df3["question_type"] == task]["ground_truth"].apply(
        lambda x: normalise_for_sklearn(x, task)).unique()
    labels = sorted(all_gt)

    for i, model in enumerate(MODELS):
        for j, fmt in enumerate(FORMATS):
            df_sub = df3[
                (df3["model"] == model) &
                (df3["question_type"] == task) &
                (df3["input_type"] == fmt)
            ].copy()

            df_sub["gt_norm"] = df_sub["ground_truth"].apply(
                lambda x: normalise_for_sklearn(x, task))
            df_sub["pred_norm"] = df_sub["parsed_answer"].fillna("unknown").apply(
                lambda x: normalise_for_sklearn(x, task))

            cm = confusion_matrix(
                df_sub["gt_norm"],
                df_sub["pred_norm"],
                labels=labels
            )

            ax = axes[i][j]
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=labels
            )
            disp.plot(ax=ax, colorbar=False,
                      xticks_rotation=45, cmap="Blues")
            ax.set_title(
                f"{model} | {fmt.upper()}",
                fontsize=9, fontweight="bold"
            )
            ax.set_xlabel("Predicted", fontsize=8)
            ax.set_ylabel("True", fontsize=8)
            ax.tick_params(axis="both", labelsize=7)

    plt.tight_layout()
    out_path = RESULTS_DIR / f"srq3_confusion_{task}.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved to srq3_confusion_{task}.png")

---
## Cell 11 — SRQ3: Per-class accuracy delta (PDF vs PNG)
For each class, how much does accuracy change between PDF and PNG?
Positive delta = PNG is better for that class.
Negative delta = PDF is better for that class.

In [ ]:
print("=" * 65)
print("SRQ3 — Per-Class Accuracy Delta (PNG acc - PDF acc)")
print("Averaged across all 3 models")
print("=" * 65)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle(
    "SRQ3 — Per-Class Accuracy Delta (PNG − PDF, avg across models)",
    fontsize=13, fontweight="bold"
)

delta_records = []

for j, task in enumerate(TASKS):
    df_task = df3[df3["question_type"] == task].copy()
    df_task["gt_norm"] = df_task["ground_truth"].apply(
        lambda x: normalise_for_sklearn(x, task))

    # Average accuracy per (class, format) across models
    class_fmt_acc = (
        df_task.groupby(["gt_norm", "input_type"])["correct"]
        .mean()
        .unstack("input_type")
    )

    # Delta: PNG - PDF
    class_fmt_acc["delta"] = (
        class_fmt_acc.get("png", pd.Series(dtype=float)) -
        class_fmt_acc.get("pdf", pd.Series(dtype=float))
    )
    class_fmt_acc = class_fmt_acc.sort_values("delta")

    print(f"\n  {task}:")
    print(f"  {'class':<22} {'pdf_acc':>8} {'png_acc':>8} {'delta':>8}")
    for cls, row in class_fmt_acc.iterrows():
        delta_records.append({
            "task": task, "class": cls,
            "pdf_acc": round(row.get("pdf", float("nan")), 4),
            "png_acc": round(row.get("png", float("nan")), 4),
            "delta":   round(row["delta"], 4)
        })
        direction = "PNG↑" if row["delta"] > 0.05 else "PDF↑" if row["delta"] < -0.05 else "~equal"
        print(f"  {cls:<22} {row.get('pdf', float('nan')):>8.3f} "
              f"{row.get('png', float('nan')):>8.3f} "
              f"{row['delta']:>+8.3f}  {direction}")

    # Plot
    ax = axes[j]
    deltas  = class_fmt_acc["delta"]
    colours = ["#2ecc71" if d > 0 else "#e74c3c" for d in deltas]
    bars    = ax.barh(deltas.index, deltas.values,
                      color=colours, edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Δ Accuracy (PNG − PDF)", fontsize=10)
    ax.set_title(task.replace("_", " ").title(), fontsize=11,
                 fontweight="bold")

    # Value labels
    for bar, val in zip(bars, deltas.values):
        xpos = val + 0.01 if val >= 0 else val - 0.01
        ha   = "left" if val >= 0 else "right"
        ax.text(xpos, bar.get_y() + bar.get_height() / 2,
                f"{val:+.2f}", va="center", ha=ha, fontsize=8)

    patches = [
        mpatches.Patch(color="#2ecc71", label="PNG better"),
        mpatches.Patch(color="#e74c3c", label="PDF better")
    ]
    ax.legend(handles=patches, fontsize=9)

plt.tight_layout()
out_path = RESULTS_DIR / "srq3_per_class_delta.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()

pd.DataFrame(delta_records).to_csv(
    RESULTS_DIR / "srq3_per_class_delta.csv", index=False)
print(f"\nSaved to srq3_per_class_delta.png and .csv")

---
## Cell 12 — SRQ3: Cross-format consistency (Definition B)
For each (movement, model, task): does the model give the same answer
on PDF as on PNG, **regardless of whether that answer is correct or not**?

High consistency = format change does not affect the model's output.
Low consistency = format change causes the model to give different answers.

In [ ]:
print("=" * 65)
print("SRQ3 — Cross-Format Consistency (Definition B)")
print("Same answer on PDF and PNG? (regardless of correctness)")
print("=" * 65)

consistency_records = []

for model in MODELS:
    print(f"\n  {model}")
    for task in TASKS:
        df_pdf = df3[
            (df3["model"] == model) &
            (df3["question_type"] == task) &
            (df3["input_type"] == "pdf")
        ].set_index("piece_id")

        df_png = df3[
            (df3["model"] == model) &
            (df3["question_type"] == task) &
            (df3["input_type"] == "png")
        ].set_index("piece_id")

        shared = df_pdf.index.intersection(df_png.index)

        pdf_norm = df_pdf.loc[shared, "parsed_answer"].fillna("").apply(
            lambda x: normalise_for_sklearn(x, task))
        png_norm = df_png.loc[shared, "parsed_answer"].fillna("").apply(
            lambda x: normalise_for_sklearn(x, task))

        consistent = (pdf_norm == png_norm).sum()
        total      = len(shared)
        rate       = consistent / total if total > 0 else float("nan")

        print(f"    {task:<20}: {consistent}/{total} = {rate:.1%}")

        consistency_records.append({
            "model":            model,
            "task":             task,
            "n_shared":         int(total),
            "n_consistent":     int(consistent),
            "consistency_rate": round(rate, 4)
        })

consistency_df = pd.DataFrame(consistency_records)

# ── SUMMARY PER MODEL (averaged across tasks) ─────────────────────────────────
print("\n  Overall consistency per model (avg across tasks):")
model_avg = consistency_df.groupby("model")["consistency_rate"].mean()
for model, rate in model_avg.items():
    print(f"    {model:<25}: {rate:.1%}")

# ── PLOT ─────────────────────────────────────────────────────────────────────
pivot = consistency_df.pivot_table(
    index="model", columns="task",
    values="consistency_rate"
)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(pivot.index))
width = 0.35
colours = ["#3498db", "#e67e22"]

for k, (col, colour) in enumerate(zip(pivot.columns, colours)):
    bars = ax.bar(x + k * width, pivot[col], width,
                  label=col.replace("_", " ").title(),
                  color=colour, alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, pivot[col]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val:.1%}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(pivot.index, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Consistency Rate (PDF == PNG)", fontsize=10)
ax.set_title(
    "SRQ3 — Cross-Format Consistency per Model\n"
    "(% of movements where PDF and PNG answers agree)",
    fontsize=11, fontweight="bold"
)
ax.axhline(1.0, color="grey", linestyle="--", alpha=0.4)
ax.legend(fontsize=9)

plt.tight_layout()
out_path = RESULTS_DIR / "srq3_consistency.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()

consistency_df.to_csv(RESULTS_DIR / "srq3_consistency.csv", index=False)
print(f"\nSaved to srq3_consistency.png and .csv")
print("\nNote: consistency measures answer agreement across formats,")
print("not correctness. A model can be consistently wrong.")

---
## All outputs saved to `results/`

| File | Content |
|------|---------|
| `srq1_summary.csv` | Accuracy + P/R/F1 per model per task |
| `srq1_confusion_matrices.png` | Confusion matrices (SRQ1) |
| `srq1_per_class_accuracy.png/.csv` | Per-class accuracy (SRQ1) |
| `srq1_agreement.csv` | Cross-model agreement table |
| `srq3_summary.csv` | Accuracy + P/R/F1 per model per format per task |
| `srq3_mcnemar.csv` | McNemar test results |
| `srq3_format_direction.csv` | Format effect direction counts |
| `srq3_confusion_key_signature.png` | Confusion matrices, key sig |
| `srq3_confusion_time_signature.png` | Confusion matrices, time sig |
| `srq3_per_class_delta.png/.csv` | Per-class PNG−PDF delta |
| `srq3_consistency.png/.csv` | Cross-format consistency |
